# Big Data Analytics Project - Data Cleaning and Feature Creation
### Group 55 – NOVA IMS
##### Spring Semester 2024/2025

## 📑 Table of Contents
- [1. Importing Libraries](#1-importing-libraries)
- [2. Loading the Dataset](#2-loading-the-dataset)
- [3. Data Cleaning: Handling Inconsistencies ](#3-data-cleaning)
- [4. Transform the features](#4-transform-features)
- [5. Create new features](#5-create-new-features)

# 1. Importing Libraries

In [0]:
%pip install holidays

Python interpreter will be restarted.
  Using cached holidays-0.73-py3-none-any.whl (954 kB)
Python interpreter will be restarted.


In [0]:
import os
import pyspark
from pyspark.sql import SparkSession
from functools import reduce
from pyspark.sql import DataFrame
import pyspark.sql.functions as F
from pyspark.sql.functions import col, when, unix_timestamp, dayofweek, udf, hour, minute, unix_timestamp, concat_ws
from IPython.display import Image, display
import matplotlib.pyplot as plt
import pandas as pd
from pyspark.sql.types import IntegerType
import holidays

# 2. Loading the Dataset

In [0]:
spark = SparkSession.builder \
    .appName("NYC Taxi 2024 Cleaning") \
    .config("spark.driver.memory", "8g") \
    .getOrCreate()

In [0]:
# Load the dataset saved after EDA
dataframe = spark.read.parquet("dbfs:/FileStore/parquets/full_dataset_after_eda.parquet")

# Quick check
dataframe.printSchema()


root
 |-- VendorID: long (nullable = true)
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: double (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: double (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: long (nullable = true)
 |-- DOLocationID: long (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- airport_fee: double (nullable = true)
 |-- month: integer (nullable = true)
 |-- year: integer (nullable = true)



In [0]:
total_size= dataframe.count()

In [0]:
dataframe.select("VendorID").distinct().show()

+--------+
|VendorID|
+--------+
|       6|
|       1|
|       2|
|       7|
|       5|
+--------+



# 3. Data Cleaning: Handling Inconsistencies


In this section, we address several data inconsistencies to ensure the reliability of downstream analyses and modeling steps. These inconsistencies include:

- ❌ Missing or null values in critical columns  
- 🚫 Invalid or extreme values (e.g., negative trip distances, zero durations with non-zero fare)  
- 🔄 Logical contradictions (e.g., drop-off time earlier than pick-up time)  
- 🧩 Out-of-range or undefined codes (e.g., unknown `RatecodeID`, undefined `VendorID`)  
- 🔢 Incorrect data types

By detecting and correcting or removing these anomalies, we aim to improve data quality and reduce noise in the dataset.


## Missing values

Since we identified approximately 6% of the records with missing values across multiple critical columns, and considering the low percentage impact and the risks associated with incorrect imputation, we decided to remove these observations. This decision aimed to ensure analytical integrity and reduce noise in the models, while still maintaining a sufficiently robust volume of data for statistical analysis and predictive modeling.

In [0]:
df = dataframe.na.drop()

## Remove useless features

### store_and_fwd_flag

In [0]:
# Count the number of rows for each value of the variable
counts = dataframe.groupBy("store_and_fwd_flag").count()

# Add a percentage column
counts_with_pct = counts.withColumn(
    "percentage",
    F.round((F.col("count") / total_size) * 100, 2)
)

counts_with_pct.show()


+------------------+---------+----------+
|store_and_fwd_flag|    count|percentage|
+------------------+---------+----------+
|              null|  6768891|      5.68|
|                 Y|   913145|      0.77|
|                 N|111454008|     93.55|
+------------------+---------+----------+



We observed that the “Y” category (which indicates a store-and-forward operation) represents a very small fraction of the dataset—less than 1% of all records. Given its extreme imbalance, this variable offers limited predictive power and is unlikely to contribute significantly to modeling or clustering tasks. Including such a sparse categorical feature could introduce unnecessary noise or complexity. Therefore, we decided to remove this variable during preprocessing to optimize the dataset and focus on more informative features.

In [0]:
df = df.drop("store_and_fwd_flag")

## Logical contradictions and Invalid or Extreme Values

### tpep_pickup_datetime and tpep_dropoff_datetime

In [0]:
invalid_rows = df.filter(F.col("tpep_pickup_datetime") > F.col("tpep_dropoff_datetime")).count()
percentage_invalid = (invalid_rows / total_size) * 100
percentage_invalid

Out[9]: 0.002085850693514718

Only 0.0021% of the records had invalid timestamps where the pickup time occurred after the dropoff time. These rows were removed due to logical inconsistency and the negligible impact of their removal on the dataset's size and representativeness.

In [0]:
# Remove rows where dropoff is earlier than or equal to pickup
df = df.filter(col("tpep_dropoff_datetime") > col("tpep_pickup_datetime"))

### passenger_count

In [0]:
invalid_rows = df.filter(col("passenger_count") <= 0).count()
percentage_invalid = (invalid_rows / total_size) * 100
percentage_invalid

Out[11]: 1.4624910660958323

Approximately 1.47% of records had an invalid `passenger_count` (≤ 0). These rows were removed as such values are not realistic and would negatively affect data quality and downstream analysis.

In [0]:
# Remove rows with 0 passengers
df = df.filter(col("passenger_count") > 0)

### trip_distance

In [0]:
# Remove rows where trip_distance is 0 or negative
df = df.filter(col("trip_distance") > 0)

In [0]:
# Cap to 100 miles 
df = df.filter(col("trip_distance") <= 100)

### payment_type

Since only 15 records had `payment_type == 5`, representing an extremely small portion of the dataset, we decided to remove these rows. The value 5 corresponds to an unknown or undefined payment type. Imputing it (e.g., with the mode) could introduce noise or bias into the data. Given the negligible impact on data volume and the unclear nature of this category, deletion was the most straightforward and analytically sound approach.

In [0]:
# Remove rows where payment_type == 5 (unkown type)
df = df.filter(col("payment_type") != 5)

### fare_amount

Rows with a `fare_amount` less than or equal to zero were removed, as such values are not realistic in the context of taxi fares and likely indicate errors or invalid entries. Additionally, we capped `fare_amount` at $200 to mitigate the influence of extreme outliers. These high values are rare and can distort analysis or model training. Given that this transformation affects only a small portion of the dataset, it helps improve overall data consistency without significant information loss.

In [0]:
# Remove rows where fare_amount is 0 or negative
df = df.filter(col("fare_amount") > 0)

In [0]:
# Cap to a maximum of 200  
df = df.filter(col("fare_amount") <= 200)

### extra

In [0]:
# Remove rows where extra is negative
df = df.filter(col("extra") >= 0)

### mta_tax

In [0]:
df = df.filter((col("mta_tax") >= 0) & (col("mta_tax") <= 0.5))

### tip_amount

Since this column only reflects credit card tips (cash tips are not included), we correct inconsistent values, remove logically invalid entries, and eliminate rows with tip amounts greater than the total paid. These steps help align the data with the official definition and prepare it for further validation and feature engineering.

In [0]:
df = df.withColumn(
    "tip_amount",
    when((col("payment_type") != 1) & (col("tip_amount") > 0), 0).otherwise(col("tip_amount"))
)

In [0]:
# Remove rows where trip_distance is 0 or negative
df = df.filter(col("tip_amount") >= 0)

In [0]:
# Remove rows where the tip amount is greater than the total amount
df = df.filter(col("total_amount") >= col("tip_amount"))

### tolls_amount

In [0]:
outliers_rows = df.filter(col("tolls_amount") > 40).count()
percentage_outliers = (outliers_rows / total_size) * 100
percentage_outliers

Out[23]: 0.0019381204230686055

To clean the `tolls_amount` field, we remove negative values, which are invalid in this context, and apply an upper cap of $40. Since only 0.0019% of the records exceed this threshold, they are considered extreme outliers and can be safely excluded without impacting the overall representativeness of the dataset.

In [0]:
df = df.filter((col("tolls_amount") >= 0) & (col("tolls_amount") <= 40))

### improvement_surcharge

In [0]:
df = df.filter(col("improvement_surcharge").isin(0.3, 1.0))

### total_amount

In [0]:
outliers_rows = df.filter(col("total_amount") > 250).count()
percentage_outliers = (outliers_rows / total_size) * 100
percentage_outliers

Out[26]: 0.0021202651315163695

We will filter the `total_amount` column to include only values between 0 and 250. Since $250 is already considered a high total fare, this threshold will help us remove extreme outliers. The proportion of records above this value is minimal, so this step will improve data consistency without significantly reducing the dataset size.

In [0]:
df = df.filter((col("total_amount") >= 0) & (col("total_amount") <= 250))

### congestion_surcharge

In [0]:
# Remove rows where trip_distance is 0 or negative
df = df.filter(col("congestion_surcharge") >= 0)

### airport_fee

We will filter the `airport_fee` column to keep only valid values: 0.0 for trips without the fee, and 1.75 for trips where the official airport surcharge was applied. This step will remove nulls or any unexpected values, helping ensure consistency and reliability in the dataset

In [0]:
df = df.filter(col("Airport_fee").isin(0.0, 1.75))

## Incorrect data types

### VendorID

We will cast the `VendorID` column from numeric to string, since this field represents a categorical identifier for the vendor and not a continuous or ordinal value. Treating it as a string helps prevent unintended numerical interpretations and ensures it is handled correctly in analysis and modeling as a categorical variable.

In [0]:
df = df.withColumn("VendorID", col("VendorID").cast("string"))

### passenger_count

We will cast the `passenger_count` column to string so that it can be treated as a categorical feature. This will allow us to group rare or unexpected values under a unified `"BIG GROUP"` category.


In [0]:
df = df.withColumn("passenger_count", col("passenger_count").cast("string"))

### PULocationID and DOLocationID

We will cast both `PULocationID` and `DOLocationID` to string, as these columns represent location identifiers rather than numerical values.

In [0]:
df = df.withColumn("PULocationID", col("PULocationID").cast("string")) \
       .withColumn("DOLocationID", col("DOLocationID").cast("string"))

### payment_type

We will cast the `payment_type` column to string because it represents categorical payment methods (e.g., credit card, cash) rather than numeric values. 

In [0]:
df = df.withColumn("payment_type", col("payment_type").cast("string"))

# 4. Transform the features

### passenger_count

In [0]:
df = df.withColumn(
    "passenger_count",
    when(col("passenger_count").isin("5","6", "7"), "BIG GROUP").otherwise(col("passenger_count"))
)

### PULocationID and DOLocationID

In [0]:
df.groupBy("PULocationID").count().orderBy("count", ascending=True).show()

+------------+-----+
|PULocationID|count|
+------------+-----+
|         110|    1|
|          84|   18|
|         176|   20|
|         187|   22|
|         204|   24|
|          99|   25|
|         105|   27|
|         245|   33|
|         115|   40|
|         199|   45|
|          30|   49|
|          27|   52|
|         109|   52|
|         172|   55|
|          59|   59|
|         251|   61|
|         206|   74|
|         111|   77|
|         156|   81|
|         221|   86|
+------------+-----+
only showing top 20 rows



In [0]:
# Count frequencies
pu_counts = df.groupBy("PULocationID").count().filter("count >= 1000")
do_counts = df.groupBy("DOLocationID").count().filter("count >= 1000")

In [0]:
# First get popular PUs
popular_pu = pu_counts.select("PULocationID").rdd.flatMap(lambda x: x).collect()
popular_do = do_counts.select("DOLocationID").rdd.flatMap(lambda x: x).collect()

# Replace rare values
df = df.withColumn("PULocationID", when(col("PULocationID").isin(popular_pu), col("PULocationID")).otherwise("other"))
df = df.withColumn("DOLocationID", when(col("DOLocationID").isin(popular_do), col("DOLocationID")).otherwise("other"))

### airport_fee

In [0]:
df = df.withColumn(
    "airport_fee",
    when(col("airport_fee") == 1.75, 1).otherwise(0)
)

In [0]:
new_size=df.count()

In [0]:
(total_size-new_size)/total_size

Out[40]: 0.12684392978501116

# 5. Create new features

## tpep_pickup_datetime and tpep_dropoff_datetime

| New Feature             | Purpose                                   |
|:-------------------------|:-----------------------------------------|
| `is_weekend`             | Captures weekend behavior                |
| `is_holiday`             | Captures holiday behavior (different patterns) |
| `pickup_hour_decimal`    | Time of day with high precision          |
| `trip_duration_minutes`  | Total time of the trip in minutes        |
| `trip_speed_mph`  | Average speed       |

In [0]:
# In Spark, dayofweek() returns 1 = Sunday, 7 = Saturday
df = df.withColumn(
    "is_weekend",
    when(dayofweek("tpep_pickup_datetime").isin(1, 7), 1).otherwise(0)
)

In [0]:
# List of US holidays (for 2024)
us_holidays = holidays.US(years=[2024])

# UDF to check if pickup date is a holiday
def is_holiday(date):
    if date is None:
        return 0
    return 1 if date.date() in us_holidays else 0

is_holiday_udf = udf(is_holiday, IntegerType())

# Apply it
df = df.withColumn("is_holiday", is_holiday_udf("tpep_pickup_datetime"))

In [0]:
df = df.withColumn(
    "pickup_hour_decimal",
    hour("tpep_pickup_datetime") + (minute("tpep_pickup_datetime") / 60)
)

In [0]:
df = df.withColumn(
    "trip_duration_minutes",
    (unix_timestamp("tpep_dropoff_datetime") - unix_timestamp("tpep_pickup_datetime")) / 60
)


In [0]:
df = df.withColumn(
    "trip_speed_mph",
    col("trip_distance") / (col("trip_duration_minutes") / 60)
)

## passenger_count

| New Feature        | Purpose                                    |
|:--------------------|:-------------------------------------------|
| `is_shared_ride`    | Flag indicating if more than 1 passenger was on the ride |

In [0]:
df = df.withColumn(
    "is_shared_ride",
    when(col("passenger_count") > 1, 1).otherwise(0)
)

## tolls_amount

| New Feature        | Purpose                                    |
|:--------------------|:-------------------------------------------|
| `has_toll`          | Flag indicating if a toll was paid during the trip |

In [0]:
df = df.withColumn(
    "has_toll",
    when(col("tolls_amount") > 0, 1).otherwise(0)
)

## PULocationID

| New Feature    | Purpose                                        |
|:---------------|:------------------------------------------------|
| `PU_DO_pair`   | Combines pickup and dropoff location IDs to capture trip origin-destination patterns |

In [0]:
# Combine pickup and dropoff location IDs
df = df.withColumn(
    "PU_DO_pair",
    concat_ws("-", col("PULocationID"), col("DOLocationID"))
)

## congestion_surcharge

| New Feature            | Purpose                                               |
|:------------------------|:------------------------------------------------------|
| `has_congestion_fee`     | Flags trips that were charged a congestion surcharge, indicating high-traffic areas or times |

In [0]:
df = df.withColumn(
    "has_congestion_fee",
    when(col("congestion_surcharge") > 0, 1).otherwise(0)
)

# Save the Dataset

In [0]:
# Repartition and save
df.repartition(200).write.mode("overwrite").parquet("/dbfs/FileStore/clean/dataset_after_cleaning.parquet")
